# What This Walkthrough Does

This capstone-style notebook puts the pieces together:

- explicit configuration
- folder structure
- API request
- raw output
- processed table
- manifest
- limitations

## Packages Used in This Notebook

This walkthrough uses:

- `requests` to collect data from the MediaWiki API
- `pandas` to create processed tables and save small JSON files
- `Path` to keep raw, processed, and report outputs separate
- `datetime` and `timezone` to timestamp the manifest
- `shutil`, `subprocess`, `sys`, and `platform` to record run evidence and version information

The point is not only to collect data, but to package the collection as a reproducible mini-project.


In [ ]:
# datetime/timezone let us create an unambiguous UTC timestamp for the manifest.
from datetime import datetime, timezone
# Path helps create output folders and file paths.
from pathlib import Path
# shutil copies already-created files into the per-run folder later in the notebook.
import shutil

# pandas turns extracted rows into tables and helps save small JSON records.
import pandas as pd
# requests sends the API request.
import requests


# Configuration

Configuration is the **plan before the run**. It contains values we choose before collection starts: endpoint, query, output location, and practical policies.

For an automatable script, many of these values would come from command-line arguments or a config file instead of being manually edited in the code. The `collector_command` should be a real terminal command someone could rerun.


In [ ]:
config = {
    "project_name": "notebook_reproducible_api_project",
    "collector_script": "notebooks/day_4_reproducible_project.ipynb",
    "collector_command": "python scripts/runnable_workflows/08_reproducible_workflow.py --config examples/configs/wikipedia_workflow.yml",
    "endpoint": "https://en.wikipedia.org/w/api.php",
    "access_route": "MediaWiki API",
    "query": "platform governance",
    "pages": 1,
    "page_size": 10,
    "outdir": "data",
    "credentials_policy": "No credentials or secrets are needed for this public API example.",
    "access_policy": "Use a descriptive User-Agent, keep requests small, and do not intensify collection without checking access rules.",
}

print(config)


# Prepare Folders

In [ ]:
outdir = Path(config["outdir"])

raw_dir = outdir / "raw"
processed_dir = outdir / "processed"
reports_dir = outdir / "reports"

for folder in [raw_dir, processed_dir, reports_dir]:
    # mkdir(..., parents=True, exist_ok=True) # creates the folder and any missing parent folders, without failing if it already exists.
    folder.mkdir(parents=True, exist_ok=True)


The folders encode workflow stages:

- `raw`: source-shaped evidence, such as API JSON or HTML
- `processed`: analysis-shaped tables
- `reports`: manifests, audits, notes, provenance

In the later run-folder section, the same idea is expanded with separate `config/` and `logs/` folders.


In [ ]:
params = {
    "action": "query",
    "list": "search",
    "srsearch": config["query"],
    "srlimit": config["page_size"],
    "format": "json",
    "formatversion": 2,
}

headers = {"User-Agent": "methodsNET-VLOP-course/1.0 reproducible project notebook"}

# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
response = requests.get(config["endpoint"], params=params, headers=headers, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
response.raise_for_status()
# .json() parses a JSON response into Python dictionaries and lists. Use it only when the response actually contains JSON.
payload = response.json()

print("Request URL:", response.url)
print("Status:", response.status_code)


# Save Raw Response

In [ ]:
raw_path = raw_dir / "notebook_project_raw_response.json"

pd.Series(
    {
        "request_url": response.url,
        "request_headers": headers,
        "status_code": response.status_code,
        "payload": payload,
    }
# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
).to_json(raw_path, indent=2)

print(raw_path)


# Process Into a Table

In [ ]:
rows = []

# .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
for item in payload.get("query", {}).get("search", []):
    rows.append(
        {
            "query": config["query"],
            "pageid": item.get("pageid"),
            "title": item.get("title"),
            "timestamp": item.get("timestamp"),
            "wordcount": item.get("wordcount"),
        }
    )

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
df = pd.DataFrame(rows)

processed_path = processed_dir / "notebook_project_processed.csv"
# to_csv(..., index=False) saves a DataFrame as a CSV file without adding pandas row numbers as a fake column.
df.to_csv(processed_path, index=False)

df.head()


# Write a Manifest

A manifest explains what was created and how. It is the **record after the run**: where the evidence is, what outputs were produced, and what limitations matter.


In [ ]:
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_name": config["project_name"],
    "collector_script": config["collector_script"],
    "collector_command": config["collector_command"],
    "access_route": config["access_route"],
    "parameters": config,
    "raw_output": str(raw_path),
    "processed_output": str(processed_path),
    "row_count": len(df),
    "limitations": [
        "Search results are API-ranked and query-dependent.",
        "This is not a measure of all platform-governance content on Wikipedia.",
        "Only one small page of results was collected for teaching purposes.",
        "No screenshots are saved because this is an API collector, not a browser collector.",
    ],
}

manifest_path = reports_dir / "notebook_project_manifest.json"
# to_json(...) saves structured data as JSON, which is useful for raw API responses and provenance records.
pd.Series(manifest).to_json(manifest_path, indent=2)

print(manifest_path)


# Practical Collection Workflows

The previous sections show a reproducible mini-project: configuration, raw data, processed data, and a manifest.

A practical collector that runs more than once needs a bit more infrastructure:

- **run folders**: one folder per execution, so outputs from different runs do not overwrite each other;
- **config snapshots**: a copy of the intended settings for this specific run;
- **logging**: what happened during a run;
- **monitoring**: whether the run produced plausible output;
- **versioning**: which code, Python version, and package versions produced the output;
- **scheduling**: when and how the collector should run.

This is the difference between "a script that works once" and "a collection workflow someone else can audit."


## Run Folders

For repeated collection, avoid writing every run into the same files. A safer pattern is one folder per run:

```text
data/runs/
  20260703T090000Z_methodsnet_course_monitor/
    config/
    raw/
    processed/
    logs/
    reports/
```

This makes it easier to compare runs, detect breakage, and reproduce what happened.

In [ ]:
# datetime/timezone create an unambiguous UTC run timestamp.
# UTC avoids confusion when work crosses time zones or daylight-saving changes.
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_notebook_demo"

run_dir = Path(config["outdir"]) / "runs" / run_id
run_raw_dir = run_dir / "raw"
run_processed_dir = run_dir / "processed"
run_logs_dir = run_dir / "logs"
run_reports_dir = run_dir / "reports"
run_config_dir = run_dir / "config"

for folder in [run_raw_dir, run_processed_dir, run_logs_dir, run_reports_dir, run_config_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print(run_dir)

## Save Run-Level Evidence

The run folder should not be empty scaffolding. It should contain the files someone would inspect later: the config snapshot, raw evidence, processed table, and reports.


In [ ]:
run_config_path = run_config_dir / "config_snapshot.json"
run_raw_path = run_raw_dir / raw_path.name
run_processed_path = run_processed_dir / processed_path.name

# Copy the plan for this run into config/.
pd.Series(config).to_json(run_config_path, indent=2)
# Copy source-shaped evidence into raw/.
shutil.copy2(raw_path, run_raw_path)
# Copy the analysis-shaped table into processed/.
shutil.copy2(processed_path, run_processed_path)

print("Config snapshot:", run_config_path)
print("Raw evidence:", run_raw_path)
print("Processed table:", run_processed_path)


## Logging

`print()` is fine for live teaching, but scheduled collectors need logs because nobody is watching the terminal.

Python's built-in `logging` module can write timestamped messages to a file. A log should record at least:

- when the run started and ended;
- which collector/config was used;
- where raw and processed outputs were written;
- how many records were collected;
- warnings about empty results, failed requests, missing fields, or selector changes;
- errors that need inspection.


In [ ]:
import logging

log_path = run_logs_dir / "run.log"

# getLogger(...) creates or retrieves a named logger. A named logger is easier
# to control than using the root logger in larger projects.
logger = logging.getLogger("notebook_practical_workflow")
logger.setLevel(logging.INFO)

# Clear old handlers so re-running this notebook cell does not duplicate messages.
logger.handlers.clear()

# FileHandler writes log messages to a file.
file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s %(levelname)s %(message)s")
)
logger.addHandler(file_handler)

logger.info("Started run %s", run_id)
logger.info("Collector script: %s", config["collector_script"])
logger.info("Config snapshot: %s", run_config_path)
logger.info("Raw evidence path: %s", run_raw_path)
logger.info("Processed output path: %s", run_processed_path)
logger.info("Collected %s processed rows", len(df))

print(log_path)


## Monitoring Breakage

Monitoring asks whether the run looks operationally plausible. It is not the same as substantive validation.

For this API example, the checks below implement the slide claims directly. They ask:

- Did the HTTP request succeed?
- Does the raw API payload still have the expected shape?
- Did the collector return at least the expected minimum number of rows?
- Are required columns present?
- Are important values missing?
- Are there duplicate IDs where we expect one row per page?
- Do numeric fields look numeric and plausible?
- Were raw and processed output files actually written?

The function writes two reports:

- `reports/monitoring_report.json` for machine-readable inspection;
- `reports/monitoring_summary.md` for a quick human-readable run note.


In [ ]:
import json


def make_check(name, passed, observed, expected, severity="error"):
    """Create one monitoring check result in a consistent shape."""
    # Every check returns the same fields. This makes the final report easy to
    # read, save as JSON, and compare across runs.
    return {
        "check": name,
        "passed": bool(passed),
        "severity": severity,
        "observed": observed,
        "expected": expected,
    }


def run_monitoring_checks(
    df,
    response,
    payload,
    config,
    raw_path,
    processed_path,
    report_dir,
    min_rows=1,
    required_columns=None,
):
    """Run operational checks and write JSON + Markdown monitoring reports."""
    # The inputs are deliberately explicit:
    # - df is the processed table we want to use later.
    # - response is the HTTP response, so we can check status codes.
    # - payload is the raw parsed API JSON, so we can check response shape.
    # - config records what we intended to collect.
    # - raw_path and processed_path prove that output files were written.
    # - report_dir is where this function writes its monitoring reports.

    # If the caller does not supply required columns, use a small default set.
    required_columns = required_columns or ["query", "pageid", "title"]

    # checks will become the detailed monitoring report.
    checks = []

    # 1. HTTP status check: catches access problems, missing pages, and rate limits.
    # A script can still produce a file after a bad request, so we check the
    # request status directly instead of assuming the output is trustworthy.
    checks.append(
        make_check(
            "http_status_ok",
            response.status_code == 200,
            response.status_code,
            200,
        )
    )

    # 2. Raw payload shape check: catches API changes or unexpected responses.
    # For this MediaWiki API example, useful results should live in
    # payload["query"]["search"]. If that structure disappears, extraction may
    # silently return zero rows.
    search_results = payload.get("query", {}).get("search") if isinstance(payload, dict) else None
    checks.append(
        make_check(
            "raw_payload_has_search_results_list",
            isinstance(search_results, list),
            type(search_results).__name__,
            "list",
        )
    )

    # 3. Row-count check: empty or tiny outputs are common signs of breakage.
    # This is intentionally simple: for a scheduled collector, zero rows should
    # be treated as suspicious until someone has inspected the raw evidence.
    checks.append(
        make_check(
            "minimum_row_count",
            len(df) >= min_rows,
            len(df),
            f">= {min_rows}",
        )
    )

    # 4. Expected page-size check: warns if the processed table and raw API
    # result count disagree. This can catch bugs where raw data exist but the
    # processing step accidentally drops rows.
    checks.append(
        make_check(
            "expected_page_size_or_explainable_shortfall",
            len(df) == min(config["page_size"], len(search_results or [])),
            len(df),
            f"same as extracted raw result count, up to requested page_size={config['page_size']}",
            severity="warning",
        )
    )

    # 5. Required-column checks: catches parser/flattening changes.
    # These are structural checks. They do not prove the data are correct, but
    # they tell us whether the table still has the shape downstream code expects.
    for column in required_columns:
        checks.append(
            make_check(
                f"required_column:{column}",
                column in df.columns,
                list(df.columns),
                column,
            )
        )

    # 6. Missing-value checks for important fields.
    # A column can exist but still be unusable if key values are missing.
    for column in ["pageid", "title"]:
        if column in df.columns:
            missing_count = int(df[column].isna().sum())
            checks.append(
                make_check(
                    f"missing_values:{column}",
                    missing_count == 0,
                    missing_count,
                    0,
                )
            )

    # 7. Duplicate-ID check: catches repeated extraction or pagination mistakes.
    # For this API search example, each pageid should appear once in this small
    # result set. Duplicates would be suspicious, not necessarily fatal.
    if "pageid" in df.columns:
        duplicate_count = int(df["pageid"].duplicated().sum())
        checks.append(
            make_check(
                "duplicate_pageids",
                duplicate_count == 0,
                duplicate_count,
                0,
                severity="warning",
            )
        )

    # 8. Numeric plausibility check: catches type/parse problems in numeric fields.
    # pd.to_numeric(..., errors="coerce") converts values that cannot be parsed
    # into NaN. Counting those NaNs tells us whether the field stayed numeric.
    if "wordcount" in df.columns:
        numeric_wordcount = pd.to_numeric(df["wordcount"], errors="coerce")
        invalid_wordcount = int(numeric_wordcount.isna().sum())
        negative_wordcount = int((numeric_wordcount < 0).sum())
        checks.append(
            make_check(
                "wordcount_numeric",
                invalid_wordcount == 0,
                invalid_wordcount,
                0,
                severity="warning",
            )
        )
        checks.append(
            make_check(
                "wordcount_non_negative",
                negative_wordcount == 0,
                negative_wordcount,
                0,
                severity="warning",
            )
        )

    # 9. Output-file checks: makes sure the run left inspectable artifacts behind.
    # This catches cases where the code created a DataFrame in memory but failed
    # to save the raw evidence or processed table to disk.
    for label, path in [("raw_output_file", raw_path), ("processed_output_file", processed_path)]:
        path = Path(path)
        size = path.stat().st_size if path.exists() else 0
        checks.append(
            make_check(
                label,
                path.exists() and size > 0,
                {"exists": path.exists(), "bytes": size},
                "file exists and is not empty",
            )
        )

    # Separate failed checks from blocking failures. A warning is worth reading,
    # but it does not necessarily mean the whole run should be rejected.
    failed_checks = [check for check in checks if not check["passed"]]
    blocking_failures = [
        check for check in failed_checks if check["severity"] == "error"
    ]

    # The JSON report is the machine-readable monitoring artifact.
    report = {
        "passed": len(blocking_failures) == 0,
        "failed_check_count": len(failed_checks),
        "blocking_failure_count": len(blocking_failures),
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_id": run_id,
        "summary": {
            "row_count": len(df),
            "column_count": len(df.columns),
            "columns": list(df.columns),
            "http_status": response.status_code,
            "raw_output": str(raw_path),
            "processed_output": str(processed_path),
        },
        "checks": checks,
    }

    # Write reports into the run folder, not just into notebook memory.
    report_dir.mkdir(parents=True, exist_ok=True)
    monitoring_path = report_dir / "monitoring_report.json"
    monitoring_summary_path = report_dir / "monitoring_summary.md"

    # JSON is useful for automated inspection and later comparison across runs.
    monitoring_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

    # Markdown is useful for humans who want a quick pass/fail summary.
    status_icon = "PASS" if report["passed"] else "FAIL"
    failed_lines = [
        f"- {check['check']} ({check['severity']}): observed {check['observed']}; expected {check['expected']}"
        for check in failed_checks
    ] or ["- None"]
    summary_text = "\n".join(
        [
            f"# Monitoring Summary: {status_icon}",
            "",
            f"Run ID: `{run_id}`",
            f"Rows: {len(df)}",
            f"HTTP status: {response.status_code}",
            f"Failed checks: {len(failed_checks)}",
            "",
            "## Failed Checks",
            *failed_lines,
            "",
            "## Files Checked",
            f"- Raw: `{raw_path}`",
            f"- Processed: `{processed_path}`",
            f"- JSON report: `{monitoring_path}`",
        ]
    )
    monitoring_summary_path.write_text(summary_text, encoding="utf-8")

    return report, monitoring_path, monitoring_summary_path


# Run the monitoring function on the outputs created above.
monitoring_report, monitoring_path, monitoring_summary_path = run_monitoring_checks(
    df=df,
    response=response,
    payload=payload,
    config=config,
    raw_path=run_raw_path,
    processed_path=run_processed_path,
    report_dir=run_reports_dir,
    min_rows=1,
    required_columns=["query", "pageid", "title", "timestamp", "wordcount"],
)

# Also report the monitoring outcome to the run log, so the log points to the
# detailed reports and records whether the run passed.
logger.info("Monitoring passed: %s", monitoring_report["passed"])
logger.info("Monitoring JSON report: %s", monitoring_path)
logger.info("Monitoring Markdown summary: %s", monitoring_summary_path)
if not monitoring_report["passed"]:
    logger.warning(
        "Monitoring failed checks: %s",
        [check["check"] for check in monitoring_report["checks"] if not check["passed"]],
    )

# Print the key report locations for the live classroom walkthrough.
print("Monitoring passed:", monitoring_report["passed"])
print("JSON report:", monitoring_path)
print("Markdown summary:", monitoring_summary_path)
print("Failed checks:", [check["check"] for check in monitoring_report["checks"] if not check["passed"]])

monitoring_report


## Versioning

A reproducible run should record not only the data and parameters, but also the code and environment state.

Useful versioning fields include:

- collector script or command;
- command-line arguments or config file;
- Git commit hash;
- Git status, because uncommitted changes matter;
- Python version and operating system;
- package versions.

This does not make the run perfectly reproducible, but it makes the conditions of the run inspectable.


In [ ]:
import platform
import subprocess
import sys


def safe_command(command):
    # subprocess.run(...) runs a terminal command from Python.
    # capture_output=True keeps stdout/stderr available as strings.
    # If a command fails, we return None instead of crashing the notebook.
    try:
        result = subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except Exception:
        return None


version_info = {
    "collector_script": config["collector_script"],
    "collector_command": config["collector_command"],
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": platform.platform(),
    "git_commit": safe_command(["git", "rev-parse", "HEAD"]),
    "git_status_short": safe_command(["git", "status", "--short"]),
    # In large projects, pip freeze can be long. For a teaching manifest, it is
    # useful because it records the package environment used for the run.
    "pip_freeze": safe_command([sys.executable, "-m", "pip", "freeze"]),
}

version_path = run_reports_dir / "version_info.json"
pd.Series(version_info).to_json(version_path, indent=2)

print("Git commit:", version_info["git_commit"])
print(version_path)


## Scheduling Collectors

Scheduling means running a collector automatically at a defined time. This can be useful for repeated data collection, but it also increases risk.

Before scheduling a scraper or API collector, ask:

- Is the access route allowed for repeated collection?
- Is the collection frequency necessary for the research question?
- Does the script respect rate limits and server load?
- Does it write logs and monitoring reports?
- Who will notice if it breaks?
- How will credentials/secrets be handled?

Three common scheduling options are cron, launchd, and GitHub Actions.

## Cron Jobs

`cron` is a scheduler on Linux/macOS. You edit scheduled commands with:

```bash
crontab -e
```

Cron syntax is:

```text
minute hour day-of-month month day-of-week command
```

Example: run every Monday at 09:00:

```cron
0 9 * * 1 cd /path/to/repo && python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data >> data/cron.log 2>&1
```

Important teaching points:

- Cron has a minimal environment. `python` may not be the same Python you use in VS Code.
- Use full paths if needed.
- Redirect output to a log file.
- Start with a low-frequency schedule.
- Do not use cron to intensify collection beyond what is justified and allowed.

## macOS launchd

On macOS, `launchd` is often more appropriate than cron. It uses `.plist` files in:

```text
~/Library/LaunchAgents/
```

A LaunchAgent can run a command at a scheduled time and send stdout/stderr to log files.

You do not need to memorize the XML. The key point is that scheduling should still call the same reproducible command-line workflow and should still write logs, monitoring reports, and manifests.

In [ ]:
launchd_example = """
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE plist PUBLIC "-//Apple//DTD PLIST 1.0//EN"
 "http://www.apple.com/DTDs/PropertyList-1.0.dtd">
<plist version="1.0">
<dict>
  <key>Label</key>
  <string>org.methodsnet.collection</string>
  <key>ProgramArguments</key>
  <array>
    <string>/bin/zsh</string>
    <string>-lc</string>
    <string>cd /path/to/repo && python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data</string>
  </array>
  <key>StartCalendarInterval</key>
  <dict>
    <key>Weekday</key><integer>1</integer>
    <key>Hour</key><integer>9</integer>
    <key>Minute</key><integer>0</integer>
  </dict>
  <key>StandardOutPath</key>
  <string>/path/to/repo/data/launchd_stdout.log</string>
  <key>StandardErrorPath</key>
  <string>/path/to/repo/data/launchd_stderr.log</string>
</dict>
</plist>
"""

print(launchd_example[:900])

## GitHub Actions

GitHub Actions can run scheduled workflows in a repository. This can be useful for public or institutional projects, but it should be used carefully:

- secrets must be stored as GitHub secrets, not in code;
- scheduled jobs should be small and justified;
- outputs/artifacts need a retention plan;
- monitoring failures should notify someone;
- platform terms/API rules still apply.

In [ ]:
github_actions_example = """
name: Scheduled collection

on:
  schedule:
    - cron: "0 7 * * 1"
  workflow_dispatch:

jobs:
  collect:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements.txt
      - run: python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data
"""

print(github_actions_example)

## Practical Wrapper Script

The repository also includes a runnable `.py` workflow for this operational layer:

```bash
python scripts/runnable_workflows/09_practical_collection_workflow.py   --run-label demo_collection   --outdir /tmp/methodsnet_practical_runs
```

It creates a run folder with:

- `config/config_snapshot.json`
- `raw/demo_raw_records.json` or `raw/input_reference.json`
- `processed/records.csv`
- `logs/run.log`
- `reports/monitoring_report.json`
- `reports/version_info.json`
- `reports/scheduling_examples.md`
- `reports/manifest.json`

This wrapper can also monitor an existing processed CSV, for example a MethodsNET scraper output.


# Exercise

Adapt the project:

1. Change the query.
2. Change the page size.
3. Add one new processed field.
4. Add one limitation to the manifest.
5. Explain what someone would need to reproduce your run.
6. Add one monitoring check that would catch a broken selector or empty API response.
7. Write a cron expression for running a collector once per week.
8. Explain why scheduling without monitoring is risky.

# Key Takeaway

A reproducible collection workflow is not just code that runs. It is a small
project structure that preserves planned settings, raw evidence, processed outputs,
logs, monitoring checks, version information, and limitations.

Scheduling is the last step, not the first. A collector should only be scheduled
after it is documented, rate-limited, monitored, and compliant with the relevant
access rules.
